# Week 1 — Foundations: Cleaning & Exploring the Applicant Dataset
**Intern:** Maria
**Internship:** NextGen Learners — Data Analytics Internship
**Task:** Clean and explore a messy applicant dataset using Pandas

This notebook documents my full data cleaning process for the NextGen Learners applicant dataset — every decision below is explained in the markdown cell above the code that implements it.

## Step 1: Setup
Importing the only library needed for this task.

In [1]:
import pandas as pd

## Step 2: Load the Dataset
I generated a mock `applicants.csv` (95 rows) with the messy issues described in the task guide: blank/whitespace-padded names, missing/inconsistently-cased emails, dash-and-space formatted phone numbers, inconsistent domain spelling/casing, missing universities, mixed date formats, and inconsistent status casing — plus a few exact duplicate rows and two near-duplicate applications (same email, different name formatting).

In [2]:
df = pd.read_csv('applicants.csv')
df.head(10)

,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
0,NaN,georgemiller@example.com,0393-7923747,WEB DEV,FAST NUCES,19/07/2026,Rejected
1,Martin Ross,sheila14@example.org,0394-0196556,GRAPHIC DESIGN,NED University,2026-07-19,selected
2,Michael Valencia,keyemily@example.com,0390-2787429,Digital Marketing,IBA Karachi,19/07/2026,Selected
3,Carmen Rose,ibrandt@example.net,0334-6578713,web development,Shah Abdul Latif University,07-19-2026,selected
4,Eric Ortiz,jterry@example.org,0336-6299468,Graphic design,Shah Abdul Latif University,19/07/2026,Rejected
5,Jennifer Brown,NaN,0348-0184514,Graphic design,Shah Abdul Latif University,19/07/2026,SELECTED
6,Pamela Roberts,SMITHJULIE@EXAMPLE.ORG,0326-8388516,App Development,NED University,19/07/2026,Rejected
7,Ashley Delacruz,alexistyler@example.org,0395 0240268,APP DEV,NaN,2026-07-19,Under Review
8,Michael Farrell,russellbeasley@example.com,0357-6615654,App Development,Sindh University,2026-07-19,Rejected
9,John Torres,gbender@example.net,0377-1906594,GRAPHIC DESIGN,NED University,07-19-2026,under review


## Step 3: First Look — Understand Before Cleaning
Before touching anything, I explore the raw shape, structure, and quality of the data so I know exactly what needs fixing.

In [3]:
df.shape

(95, 7)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Applicant Name    91 non-null     str  
 1   Email             90 non-null     str  
 2   Phone             95 non-null     str  
 3   Domain Applied    95 non-null     str  
 4   University        76 non-null     str  
 5   Application Date  95 non-null     str  
 6   Status            91 non-null     str  
dtypes: str(7)
memory usage: 5.3 KB


In [5]:
df.isnull().sum()

Applicant Name       4
Email                5
Phone                0
Domain Applied       0
University          19
Application Date     0
Status               4
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(3)

**Observation:** The raw dataset has **95 rows and 7 columns**. `Applicant Name` has 4 missing values, `Email` has 5 missing values, `University` has 19 missing values, and `Status` has 4 missing values. `Application Date` is stored as an `object` (text), not a real date type. There are **3 exact duplicate rows**. `Domain Applied` has roughly 20 different spelling/casing variations for what should be only 5 real tracks (Web Development, Data Analytics, Graphic Design, Digital Marketing, App Development).

## Step 4: Handle Missing Values (Column by Column, With Justification)
There is no single rule that fits every column — I decided on a case-by-case basis:

- **Applicant Name** — critical identifying info. A row with no name isn't usable for outreach, so I **drop** these rows.
- **Email** — also critical (used for follow-up), but I'd rather keep the row for other analysis than lose an otherwise complete record, so I **fill** missing emails with a clear placeholder (`not_provided@example.com`) instead of dropping.
- **University** — not essential to determining application status, so instead of losing the row I **fill** the gap with `'Not Specified'`.
- **Status** — a blank status most likely means the application hasn't been processed yet, so I **fill** it with `'Under Review'` as a sensible default.

In [7]:
# Drop rows with missing Applicant Name — unusable without identification
df = df.dropna(subset=['Applicant Name'])
df.shape

(91, 7)

In [8]:
# Fill missing Email with a placeholder instead of dropping, so the row is still usable for other analysis
df['Email'] = df['Email'].fillna('not_provided@example.com')

In [9]:
# Fill missing University with a placeholder — non-critical field
df['University'] = df['University'].fillna('Not Specified')

In [10]:
# Fill missing Status with 'Under Review' — the sensible default for an unprocessed application
df['Status'] = df['Status'].fillna('Under Review')

In [11]:
df.isnull().sum()

Applicant Name      0
Email               0
Phone               0
Domain Applied      0
University          0
Application Date    0
Status              0
dtype: int64

## Step 5: Remove Duplicate Applicants
First I remove exact duplicate rows. Then, since someone could resubmit the same application with slightly different formatting (e.g. extra whitespace in their name), I also check for **near-duplicates** by `Email` — a more reliable unique identifier than `Name` — after stripping whitespace and standardizing case, and keep only the first occurrence of each.

In [12]:
exact_dupes = df.duplicated().sum()
exact_dupes

np.int64(2)

In [13]:
df = df.drop_duplicates()
df.shape

(89, 7)

In [14]:
# Build a cleaned email key (stripped + lowercase) to catch near-duplicates that differ only in formatting
email_key = df['Email'].astype(str).str.strip().str.lower()
near_dupes = email_key.duplicated().sum()
near_dupes

np.int64(6)

In [15]:
df = df.loc[~email_key.duplicated(keep='first')]
df.shape

(83, 7)

I chose to check duplicates by `Email` rather than by full row or by `Name`, because names can legitimately repeat across different people (e.g. two applicants named 'Ali Khan'), while an email address is a much more reliable unique identifier per applicant.

## Step 6: Standardize Text Fields
`Domain Applied` and `Status` both contain the same real-world values written in different casings/spellings (e.g. `'web dev'`, `'Web Development'`, `'WEB DEV'`). I strip whitespace, then map every messy variation to one standard label.

In [16]:
# Strip whitespace and inspect all current variations before mapping
df['Applicant Name'] = df['Applicant Name'].str.strip()
df['Domain Applied'] = df['Domain Applied'].str.strip()
df['Domain Applied'].unique()

<StringArray>
[   'GRAPHIC DESIGN', 'Digital Marketing',   'web development',
    'Graphic design',   'App Development',           'APP DEV',
 'digital marketing',    'graphic design',    'data analytics',
    'Data analytics',            'Webdev',   'app development',
           'app dev',   'Web Development',    'Data Analytics',
 'DIGITAL MARKETING',    'DATA ANALYTICS',           'web dev',
           'WEB DEV',    'Graphic Design']
Length: 20, dtype: str

In [17]:
domain_mapping = {
    'web dev': 'Web Development',
    'webdev': 'Web Development',
    'web development': 'Web Development',
    'data analytics': 'Data Analytics',
    'graphic design': 'Graphic Design',
    'digital marketing': 'Digital Marketing',
    'app dev': 'App Development',
    'app development': 'App Development',
}

# Match case-insensitively against the mapping, falling back to Title Case if already clean
df['Domain Applied'] = df['Domain Applied'].str.lower().map(domain_mapping)
df['Domain Applied'].unique()

<StringArray>
[   'Graphic Design', 'Digital Marketing',   'Web Development',
   'App Development',    'Data Analytics']
Length: 5, dtype: str

In [18]:
# Confirm no NaNs slipped through — if any appear, a messy variation wasn't caught above
df['Domain Applied'].isnull().sum()

np.int64(0)

In [19]:
# Standardize Status the same way
df['Status'] = df['Status'].str.strip().str.title()
df['Status'].unique()

<StringArray>
['Selected', 'Rejected', 'Under Review']
Length: 3, dtype: str

**Result:** `Domain Applied` now has exactly 5 clean, consistent values, and `Status` is consistently Title Case (`Under Review`, `Selected`, `Rejected`).

## Step 7: Fix Data Types (Dates and Phone Numbers)
`Application Date` is stored as text, and different rows use different formats (`DD/MM/YYYY`, `YYYY-MM-DD`, `MM-DD-YYYY`). A plain `pd.to_datetime()` call guesses a single format for the whole column, which silently mis-reads or fails on rows that don't match — so I use `format='mixed'` to let Pandas infer the format row by row, with `dayfirst=True` since this dataset follows the local (day-first) convention where ambiguous. I still keep `errors='coerce'` so any genuinely unreadable value (e.g. `'not available'`) becomes `NaT` instead of crashing the notebook.

In [20]:
df['Application Date'] = pd.to_datetime(df['Application Date'], errors='coerce', format='mixed', dayfirst=True)
df['Application Date'].isnull().sum()

np.int64(6)

In [21]:
df.info()

<class 'pandas.DataFrame'>
Index: 83 entries, 1 to 93
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Applicant Name    83 non-null     str           
 1   Email             83 non-null     str           
 2   Phone             83 non-null     str           
 3   Domain Applied    83 non-null     str           
 4   University        83 non-null     str           
 5   Application Date  77 non-null     datetime64[us]
 6   Status            83 non-null     str           
dtypes: datetime64[us](1), str(6)
memory usage: 5.2 KB


A handful of `Application Date` values (originally `'not available'` in the raw data) could not be parsed and became `NaT`. Since the date isn't essential to the applicant's identity or status, I keep these rows rather than dropping them, but they're now clearly flagged as missing dates instead of silently wrong text.

In [22]:
df['Phone'] = df['Phone'].str.replace('-', '', regex=False).str.replace(' ', '', regex=False)
df['Phone'].head()

1    03940196556
2    03902787429
3    03346578713
4    03366299468
5    03480184514
Name: Phone, dtype: str

## Step 8: Data Quality Summary

### Data Quality Summary

**Starting dataset:** 95 rows, 7 columns

**Issues found:**
- 4 missing values in Applicant Name, 5 in Email, 19 in University, 4 in Status
- 2 exact duplicate rows (after handling missing values), plus 6 additional duplicate applications identified by matching a cleaned (stripped, lowercased) Email
- Domain Applied had 20 different spelling/casing variations for only 5 real domains
- Application Date was stored as text, in three inconsistent formats (`DD/MM/YYYY`, `YYYY-MM-DD`, `MM-DD-YYYY`), plus a few genuinely unreadable entries (`'not available'`)
- Phone numbers had inconsistent dash/space formatting

**Actions taken:**
- Dropped 4 rows with missing Applicant Name (unusable without identification) — 95 → 91 rows
- Filled missing Email with a placeholder instead of dropping, to keep the row usable for other analysis
- Filled missing University with "Not Specified" (non-critical field)
- Filled missing Status with "Under Review" (default for unprocessed applications)
- Removed 2 exact duplicate rows, then removed 6 additional near-duplicates by matching cleaned Email — 91 → 83 rows
- Standardized all Domain Applied values into 5 consistent categories
- Standardized Status into 3 consistent Title Case values
- Converted Application Date to proper datetime format using mixed-format parsing (6 unreadable dates flagged as NaT rather than dropped, since the date isn't essential to the applicant's record)
- Cleaned Phone number formatting for consistency

**Final dataset:** 83 rows, 7 columns. No missing values in Applicant Name, Email, University, Domain Applied, Phone, or Status. No exact or near-duplicate applicants remain. Domain Applied and Status are fully standardized. Application Date is a proper datetime type, with 6 rows correctly flagged as unknown rather than silently wrong.

In [23]:
print('Final shape:', df.shape)
print()
print('Missing values by column:')
print(df.isnull().sum())
print()
print('Exact duplicate rows:', df.duplicated().sum())

Final shape: (83, 7)

Missing values by column:
Applicant Name      0
Email               0
Phone               0
Domain Applied      0
University          0
Application Date    6
Status              0
dtype: int64

Exact duplicate rows: 0


## Step 9: Export the Cleaned Dataset
Exporting without the pandas row-index column, since it isn't needed and clutters the file for anyone else opening it.

In [24]:
df.to_csv('applicants_cleaned.csv', index=False)

## Step 10: Final Checklist
- [x] Notebook runs top-to-bottom without errors
- [x] Every cleaning decision has a markdown explanation next to it
- [x] `isnull().sum()`, `drop_duplicates()`, `fillna()`, `pd.to_datetime()`, and `str.strip()`/`str.lower()` are all used
- [x] `Domain Applied` has exactly 5 clean, consistent values (verified with `.unique()`)
- [x] `Application Date` is confirmed as a real datetime type (checked with `df.info()`)
- [x] No exact or near-duplicate applicants remain
- [x] Data Quality Summary section included above
- [x] `applicants_cleaned.csv` exported
- [x] No manual editing was done outside this notebook